# Home Credit — Data Analysis & Model Evaluation

Notebook gồm 5 phần chính:
1. **Setup** — kết nối DuckDB, tổng quan dữ liệu
2. **Đánh giá chất lượng dữ liệu** — class balance, null rate, correlation
3. **EDA** — default rate theo các feature chính
4. **Train models** — LightGBM + LR Scorecard
5. **Đánh giá model** — ROC, confusion matrix, feature importance, calibration

## 1. Setup & Overview

In [2]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)

tables = con.execute('''
SELECT 
    t.table_schema as schema,
    t.table_name as name,
    COUNT(c.column_name) as column_count
FROM information_schema.tables t
LEFT JOIN information_schema.columns c 
    ON t.table_schema = c.table_schema 
    AND t.table_name = c.table_name
WHERE t.table_schema NOT IN ('information_schema', 'system')
GROUP BY t.table_schema, t.table_name
ORDER BY schema, name
''').df()

tables['row_count'] = tables.apply(
    lambda r: con.execute(f"SELECT COUNT(*) FROM {r['schema']}.{r['name']}").fetchone()[0],
    axis=1
)
print('Connected ✓')
tables

ModuleNotFoundError: No module named 'duckdb'

## 2. Đánh giá chất lượng dữ liệu

4 chart trong 1 hình:
- (top-left) Class balance — tỷ lệ default vs no-default
- (top-right) Null rate per feature (silver)
- (bottom-left) Correlation với target (gold features)
- (bottom-right) Heatmap top features

In [ ]:
# Class balance
target_dist = con.execute(
    'SELECT is_default, COUNT(*) AS n FROM silver.home_credit_cleansed GROUP BY is_default ORDER BY is_default'
).df()

# Null rate per silver feature
silver_cols = con.execute(
    "SELECT column_name FROM information_schema.columns "
    "WHERE table_schema='silver' AND table_name='home_credit_cleansed'"
).df()['column_name'].tolist()
null_query = 'SELECT ' + ', '.join(
    f'ROUND(100.0 * AVG(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END), 2) AS "{c}"'
    for c in silver_cols
) + ' FROM silver.home_credit_cleansed'
null_df = con.execute(null_query).df().T.reset_index()
null_df.columns = ['feature', 'null_pct']
null_df = null_df.sort_values('null_pct', ascending=True).tail(15)

# Correlation với target (gold)
gold_cols = ['credit_score_midpoint', 'debt_to_income_ratio', 'loan_amount_to_income',
             'log_monthly_income', 'rating_ordinal', 'num_previous_loans',
             'previous_default_rate', 'num_bureau_records', 'num_active_credit',
             'ext_source_1', 'ext_source_3', 'age_years', 'education_ordinal',
             'is_married_flag', 'gender_male_flag', 'is_default']
gold_df = con.execute(f"SELECT {', '.join(gold_cols)} FROM gold.hc_features_v1").df()
corr = gold_df.corr()['is_default'].drop('is_default').sort_values()
top_feats = corr.abs().sort_values(ascending=False).head(8).index.tolist() + ['is_default']

# ── Plot 4-panel figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# (1) Class balance pie
axes[0, 0].pie(target_dist['n'],
               labels=[f"No Default\n{target_dist.iloc[0]['n']:,}",
                       f"Default\n{target_dist.iloc[1]['n']:,}"],
               colors=['#5DADE2', '#E74C3C'], autopct='%1.2f%%',
               wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ratio = target_dist.iloc[0]['n'] // target_dist.iloc[1]['n']
axes[0, 0].set_title(f'Class balance — imbalance ~{ratio}:1', fontweight='bold')

# (2) Null rate top 15
colors = ['#7ED957' if v < 5 else '#F5B041' if v < 30 else '#E74C3C' for v in null_df['null_pct']]
axes[0, 1].barh(null_df['feature'], null_df['null_pct'], color=colors, edgecolor='black', linewidth=0.4)
axes[0, 1].axvline(5,  color='#F5B041', linestyle='--', alpha=0.5)
axes[0, 1].axvline(30, color='#E74C3C', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Null rate (%)')
axes[0, 1].set_title('Null rate per feature (top 15)', fontweight='bold')

# (3) Correlation với target
corr_colors = ['#E74C3C' if v > 0 else '#5DADE2' for v in corr.values]
axes[1, 0].barh(corr.index, corr.values, color=corr_colors, edgecolor='black', linewidth=0.4)
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_xlabel('Correlation với is_default')
axes[1, 0].set_title('Feature ↔ Target (đỏ: risk ↑, xanh: an toàn ↑)', fontweight='bold')

# (4) Heatmap top features
cm = gold_df[top_feats].corr()
mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=False,
            cbar_kws={'shrink': 0.8}, ax=axes[1, 1], annot_kws={'fontsize': 8})
axes[1, 1].set_title('Correlation heatmap — Top 8 features', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n→ Imbalance {ratio}:1 — cần is_unbalance=True khi train')
print(f'→ Cột null cao nhất: {null_df.tail(3)[["feature","null_pct"]].to_dict(orient="records")}')

## 3. EDA — Default rate theo các feature chính

4 chart trong 1 hình: credit score · DTI · age · education

In [1]:
# Tính 4 bảng aggregated
score_df = con.execute('''
    SELECT (credit_score_range_lower + credit_score_range_upper) / 2.0 AS score, is_default
    FROM silver.home_credit_cleansed
    WHERE credit_score_range_lower IS NOT NULL
''').df()

dti_df = con.execute('''
    SELECT
        CASE
          WHEN debt_to_income_ratio < 0.1 THEN '< 0.1'
          WHEN debt_to_income_ratio < 0.2 THEN '0.1–0.2'
          WHEN debt_to_income_ratio < 0.3 THEN '0.2–0.3'
          WHEN debt_to_income_ratio < 0.5 THEN '0.3–0.5'
          WHEN debt_to_income_ratio < 1.0 THEN '0.5–1.0'
          ELSE '> 1.0'
        END AS bucket,
        ROUND(AVG(is_default) * 100, 2) AS default_pct,
        COUNT(*) AS n
    FROM gold.hc_features_v1
    GROUP BY bucket
    ORDER BY MIN(debt_to_income_ratio)
''').df()

age_df = con.execute('''
    SELECT
        CASE
          WHEN age_years < 25 THEN '18-24'
          WHEN age_years < 35 THEN '25-34'
          WHEN age_years < 45 THEN '35-44'
          WHEN age_years < 55 THEN '45-54'
          WHEN age_years < 65 THEN '55-64'
          ELSE '65+'
        END AS bucket,
        ROUND(AVG(is_default) * 100, 2) AS default_pct
    FROM gold.hc_features_v1
    GROUP BY bucket ORDER BY bucket
''').df()

edu_map = {1: 'Lower\nsec', 2: 'Secondary', 3: 'Incomplete\nhigher', 4: 'Higher', 5: 'Academic'}
edu_df = con.execute('''
    SELECT education_ordinal, ROUND(AVG(is_default) * 100, 2) AS default_pct
    FROM gold.hc_features_v1 WHERE education_ordinal IS NOT NULL
    GROUP BY education_ordinal ORDER BY education_ordinal
''').df()
edu_df['label'] = edu_df['education_ordinal'].map(edu_map)

# ── Plot 4-panel figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# (1) Credit score density by class
sns.histplot(data=score_df, x='score', hue='is_default', bins=40,
             palette=['#5DADE2', '#E74C3C'], alpha=0.7,
             stat='density', common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title('Phân phối credit score theo class', fontweight='bold')
axes[0, 0].set_xlabel('Credit Score')

# (2) DTI buckets
bars = axes[0, 1].bar(dti_df['bucket'], dti_df['default_pct'], color='#E74C3C',
                       alpha=0.75, edgecolor='black')
axes[0, 1].set_title('Default rate theo DTI bucket', fontweight='bold')
axes[0, 1].set_ylabel('Default rate (%)')
for b, v in zip(bars, dti_df['default_pct']):
    axes[0, 1].text(b.get_x() + b.get_width()/2, v + 0.2, f'{v}%',
                    ha='center', fontweight='bold', fontsize=9)

# (3) Age buckets
bars = axes[1, 0].bar(age_df['bucket'], age_df['default_pct'], color='#3498DB',
                       alpha=0.75, edgecolor='black')
axes[1, 0].set_title('Default rate theo độ tuổi', fontweight='bold')
axes[1, 0].set_ylabel('Default rate (%)')
for b, v in zip(bars, age_df['default_pct']):
    axes[1, 0].text(b.get_x() + b.get_width()/2, v + 0.1, f'{v}%',
                    ha='center', fontweight='bold', fontsize=9)

# (4) Education
bars = axes[1, 1].bar(edu_df['label'], edu_df['default_pct'], color='#16A085',
                       alpha=0.75, edgecolor='black')
axes[1, 1].set_title('Default rate theo học vấn', fontweight='bold')
axes[1, 1].set_ylabel('Default rate (%)')
for b, v in zip(bars, edu_df['default_pct']):
    axes[1, 1].text(b.get_x() + b.get_width()/2, v + 0.1, f'{v}%',
                    ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

NameError: name 'con' is not defined

## 4. Train models

Train cả 2 model trực tiếp từ notebook (mỗi cái ~10–15s).

In [ ]:
# DuckDB không cho phép cùng 1 file mở với 2 mode (read-only vs read-write) trong cùng process.
# Đóng `con` (read-only) trước khi train (cần read-write qua SQLAlchemy).
try:
    con.close()
except Exception:
    pass

from ml.retrain_customer_model import train as train_lgbm
from ml.train_scorecard       import train as train_scorecard

train_lgbm()
print('\n' + '='*60 + '\n')
train_scorecard()

# Lưu ý: sau cell này, `con` đã đóng. Nếu muốn dùng `con` lại → restart kernel + chạy từ đầu.

## 5. Đánh giá Model

Load 2 artifact đã train, build lại test set, đánh giá 6 chỉ tiêu trong 1 figure:
- ROC curve · Confusion matrix · Feature importance · Score distribution · Calibration · Precision-Recall

In [ ]:
import joblib
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve

from ml.retrain_customer_model import _QUERY as LGBM_QUERY, ALL_FEATURES as LGBM_FEATURES
from ml.train_scorecard       import QUERY as SCORE_QUERY, ALL_FEATURES as SCORE_FEATURES, prob_to_score
from utils.db_connection      import get_engine

lgbm_art      = joblib.load(ROOT / 'ml' / 'models' / 'customer_risk_model.pkl')
scorecard_art = joblib.load(ROOT / 'ml' / 'models' / 'scorecard_model.pkl')
engine        = get_engine()

# ── Test set cho LightGBM ───────────────────────────────────────────────
df_l = pd.read_sql(LGBM_QUERY, engine)
for c in ['ext_source_1','ext_source_3','gender_male_flag','education_ordinal',
          'age_years','cnt_children','cnt_fam_members','is_married_flag']:
    df_l[c] = df_l[c].fillna(df_l[c].median())
df_l['employment_status'] = df_l['employment_status'].fillna('Other/Unknown')
df_l = df_l.dropna(subset=['is_default','monthly_income','loan_amount','credit_score'])
_, Xl_te, _, yl_te = train_test_split(
    df_l[LGBM_FEATURES], df_l['is_default'],
    test_size=0.2, random_state=42, stratify=df_l['is_default']
)
y_prob_lgbm = lgbm_art['pipeline'].predict_proba(Xl_te)[:, 1]

# ── Test set cho LR Scorecard ──────────────────────────────────────────
df_s = pd.read_sql(SCORE_QUERY, engine)
df_s[SCORE_FEATURES[:-1]] = df_s[SCORE_FEATURES[:-1]].fillna(df_s[SCORE_FEATURES[:-1]].median(numeric_only=True))
df_s['employment_status_grouped'] = df_s['employment_status_grouped'].fillna('Other/Unknown')
df_s = df_s.dropna(subset=['is_default'])
_, Xs_te, _, ys_te = train_test_split(
    df_s[SCORE_FEATURES], df_s['is_default'],
    test_size=0.2, random_state=42, stratify=df_s['is_default']
)
y_prob_score = scorecard_art['pipeline'].predict_proba(Xs_te)[:, 1]

auc_lgbm  = roc_auc_score(yl_te, y_prob_lgbm)
auc_score = roc_auc_score(ys_te, y_prob_score)
print(f'LightGBM    AUC = {auc_lgbm:.4f}')
print(f'LR Scorecard AUC = {auc_score:.4f}')

# ── Plot 6-panel figure ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# (1) ROC
fpr_l, tpr_l, _ = roc_curve(yl_te, y_prob_lgbm)
fpr_s, tpr_s, _ = roc_curve(ys_te, y_prob_score)
axes[0, 0].plot(fpr_l, tpr_l, color='#27AE60', linewidth=2.5, label=f'LightGBM ({auc_lgbm:.3f})')
axes[0, 0].plot(fpr_s, tpr_s, color='#8E44AD', linewidth=2.5, label=f'LR Score ({auc_score:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0, 0].set_xlabel('FPR'); axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve', fontweight='bold'); axes[0, 0].legend(loc='lower right')

# (2) Confusion matrix - LightGBM
cm_l = confusion_matrix(yl_te, (y_prob_lgbm >= 0.5).astype(int))
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 1],
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Act 0', 'Act 1'],
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})
tn, fp, fn, tp = cm_l.ravel()
recall = tp / (tp + fn) if (tp + fn) else 0
axes[0, 1].set_title(f'Confusion Matrix — LightGBM\nRecall(default) = {recall:.2%}',
                     fontweight='bold')

# (3) Feature importance
lgbm_clf = lgbm_art['pipeline'].named_steps['classifier']
imp = pd.DataFrame({'feat': LGBM_FEATURES, 'imp': lgbm_clf.feature_importances_})
imp = imp.sort_values('imp', ascending=True).tail(15)
axes[0, 2].barh(imp['feat'], imp['imp'], color='#16A085', edgecolor='black', linewidth=0.4)
axes[0, 2].set_xlabel('Split count'); axes[0, 2].set_title('Top 15 Feature Importance (LightGBM)', fontweight='bold')

# (4) Score distribution
test_scores = prob_to_score(y_prob_score)
score_dist  = pd.DataFrame({'score': test_scores, 'is_default': ys_te.values})
sns.histplot(data=score_dist, x='score', hue='is_default', bins=40,
             palette=['#5DADE2', '#E74C3C'], alpha=0.7,
             stat='density', common_norm=False, ax=axes[1, 0])
axes[1, 0].set_title(f'Credit Score Distribution\n[{test_scores.min()}–{test_scores.max()}], mean = {test_scores.mean():.0f}',
                     fontweight='bold')
axes[1, 0].set_xlabel('Credit Score (300–850)')

# (5) Calibration
fp_l, mp_l = calibration_curve(yl_te, y_prob_lgbm,  n_bins=10, strategy='quantile')
fp_s, mp_s = calibration_curve(ys_te, y_prob_score, n_bins=10, strategy='quantile')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect')
axes[1, 1].plot(mp_l, fp_l, 'o-', color='#27AE60', linewidth=2, markersize=8, label='LightGBM')
axes[1, 1].plot(mp_s, fp_s, 's-', color='#8E44AD', linewidth=2, markersize=8, label='LR Score')
axes[1, 1].set_xlabel('Predicted prob'); axes[1, 1].set_ylabel('Actual rate')
axes[1, 1].set_title('Calibration Plot', fontweight='bold'); axes[1, 1].legend(loc='upper left')

# (6) Precision-Recall
pr_l, rc_l, _ = precision_recall_curve(yl_te, y_prob_lgbm)
pr_s, rc_s, _ = precision_recall_curve(ys_te, y_prob_score)
ap_l = average_precision_score(yl_te, y_prob_lgbm)
ap_s = average_precision_score(ys_te, y_prob_score)
axes[1, 2].plot(rc_l, pr_l, color='#27AE60', linewidth=2.5, label=f'LightGBM (AP={ap_l:.3f})')
axes[1, 2].plot(rc_s, pr_s, color='#8E44AD', linewidth=2.5, label=f'LR Score (AP={ap_s:.3f})')
axes[1, 2].axhline(yl_te.mean(), color='k', linestyle='--', alpha=0.4,
                   label=f'Baseline ({yl_te.mean():.3f})')
axes[1, 2].set_xlabel('Recall'); axes[1, 2].set_ylabel('Precision')
axes[1, 2].set_title('Precision-Recall Curve', fontweight='bold'); axes[1, 2].legend(loc='upper right')

plt.tight_layout()
plt.show()

## Kết luận

**Data:**
- 300k applications, class imbalance ~11:1 (8% default)
- `EXT_SOURCE_1` null 56% → fill median
- Feature mạnh nhất: `credit_score_midpoint`, `ext_source_3`, `num_bureau_records`, `age_years`

**Models:**
- LightGBM AUC ~0.75 — dùng cho `/applications/submit`
- LR Scorecard AUC ~0.73 — dùng cho `/credit-score/{id}` với FICO 300–850
- Calibration của LightGBM cần điều chỉnh khi triển khai

**Bước tiếp:**
- Optuna tuning → +0.01–0.015 AUC
- Tích hợp vào backend endpoint